In [4]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [5]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}
@tool
def get_product(name:str)->str:
    """Look up a product by name and return its price, rating, stock, and description."""
    product = PRODUCTS.get(name.lower())
    if product:
        return f"Product: {name}\nPrice: ${product['price']}\nRating: {product['rating']} stars\nDescription: {product['description']}"
    else:
        return f"Product '{name}' not found."

In [6]:
llm=ChatGroq(model="openai/gpt-oss-20b", temperature=0)
agent=create_agent(
    llm,
    tools=[get_product],
    system_prompt="you are a helpful product assistant fro online tech store."

)


In [7]:
def ask(question:str)->str:
    result=agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [8]:
ask("What is the price and rating of wireless headphones?")

**Wireless Headphones**  
- **Price:** $79.99  
- **Rating:** 4.6 ★  

Let me know if you’d like more details or help with anything else!


In [9]:
# 1. extract product name from question 
# 2 . call get_product with the extracted product name
# 3. analys the ans {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."}
# 4. return human readable answer

In [10]:
REVIEW = {
    "wireless headphones": {
        "review": 1262,
        "rating": 4.6,
        "description": "Great sound quality and battery life."
    },
    "smart watch": {
        "review": 854,
        "rating": 4.3,
        "description": "Accurate tracking and comfortable to wear."
    },
    "mechanical keyboard": {
        "review": 432,
        "rating": 4.8,
        "description": "Excellent typing experience and build quality."
    },
    "laptop stand": {
        "review": 210,
        "rating": 4.1,
        "description": "Sturdy construction and adjustable height."
    },
}
@tool
def get_review(name:str) -> str:
    """ Look up a product review by name and return its review count, rating, and description."""
    review = REVIEW.get(name.lower())
    if review:
        return f"Product: {name}\nReview Count: {review['review']}\nRating: {review['rating']} stars\nDescription: {review['description']}"
    else:
        return f"Review for product '{name}' not found."

In [14]:

llm_gpt = ChatGroq(model="openai/gpt-oss-20b", temperature=0)


agent_gpt = create_agent(
    llm_gpt,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store. You can provide product details and reviews. in proper human readable format."
)


def ask_gpt(question: str) -> str:
    result= agent_gpt.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)
    print("result: "+str(result))


In [15]:
ask_gpt("What is the price and rating of wireless headphones?")

**Wireless Headphones**

- **Price:** $79.99  
- **Rating:** 4.6 ★ (out of 5)
result: {'messages': [HumanMessage(content='What is the price and rating of wireless headphones?', additional_kwargs={}, response_metadata={}, id='0959967c-dcc7-483b-a767-488cb64d26ab'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_product with name "wireless headphones".', 'tool_calls': [{'id': 'fc_342c94ce-6fde-40e0-9905-646b3227a127', 'function': {'arguments': '{"name":"wireless headphones"}', 'name': 'get_product'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 203, 'total_tokens': 242, 'completion_time': 0.068474979, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.010639318, 'prompt_tokens_details': None, 'queue_time': 0.307783577, 'total_time': 0.079114297}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4a35f7bd1b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls'

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm_gemma = ChatGoogleGenerativeAI(
    model="gemma-4-31b-it"
)

agent_gemma = create_agent(
    llm_gemma,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store. You can provide product details and reviews. in proper human readable format."
)
def ask_gemma(question: str) -> str:
    result= agent_gemma.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)
    print("result: "+str(result))


In [17]:
ask_gemma("What is the price and rating of wireless headphones?")

The wireless headphones are priced at $79.99 and have a rating of 4.6 stars.
result: {'messages': [HumanMessage(content='What is the price and rating of wireless headphones?', additional_kwargs={}, response_metadata={}, id='99cdd199-54fb-4a3e-81c6-b29bfe80e9d6'), AIMessage(content=[{'type': 'thinking', 'thinking': 'The user is asking for the price and rating of "wireless headphones".\nLooking at the available tools, `get_product` takes a product name and returns its price, rating, stock, and description.\nI should call `get_product` with the argument `name=\'wireless headphones\'`.'}], additional_kwargs={'function_call': {'name': 'get_product', 'arguments': '{"name": "wireless headphones"}'}, '__gemini_function_call_thought_signatures__': {'call_313715': 'EiYKJGUyNDgzMGE3LTVjZDYtNDJmZS05OThiLWVlNTM5ZTcyYjljMw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemma-4-31b-it', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1335c-6ac2-7e90-81a9-d5c4

In [18]:
print("ask_gemma: how do people like smart watch?")
ask_gemma("how do people like smart watch?")
print("----------------------------------------------")
print("ask_gpt: how do people like smart watch?")
ask_gpt("how do people like smart watch?")


ask_gemma: how do people like smart watch?
People generally like smart watches, with an average rating of **4.3 stars** based on 854 reviews. Users frequently mention that they are comfortable to wear and provide accurate tracking.
result: {'messages': [HumanMessage(content='how do people like smart watch?', additional_kwargs={}, response_metadata={}, id='febdc641-2331-45b2-bc24-b7eb4fc3f5b9'), AIMessage(content=[{'type': 'thinking', 'thinking': 'The user is asking "how do people like smart watch?". This is a general question about product sentiment/reviews for "smart watch".\nI have a tool `get_review` that can provide review counts, ratings, and descriptions for a product.\nHowever, "smart watch" is a category, not a specific product name.\nI should check if there are specific smart watches available or if I can get a general review for "smart watch".\nLooking at the tool definition: `get_review(name: STRING)`.\nIf I call `get_review(name="smart watch")`, it might return general revi